# Estrazione fustelle MySecretCase → Google Sheet
Task 2 — pipeline PDF vettoriale → immagine → estrazione vision → Google Sheet

**Perché serve la vision:** i PDF delle fustelle sono export Illustrator con il testo
convertito in tracciati vettoriali. `pdfplumber`/`PyPDF2` tornano ~16 caratteri per
pagina. L'unica strada affidabile è renderizzare ogni pagina come immagine e farla
leggere a un modello vision con un prompt strutturato.

**Come usare questo notebook:**
1. Esegui le celle in ordine (Runtime → Esegui tutto, oppure una alla volta).
2. Alla cella 3 ti verrà chiesto di caricare i PDF (drag & drop o selezione multipla).
3. Alla cella 4 inserisci la tua chiave API Claude (non viene salvata da nessuna parte).
4. L'ultima cella scrive su Google Sheet — ti chiederà di autenticarti col tuo
   account Google al volo, nessun file di credenziali da preparare.


## 1. Installazione dipendenze

In [ ]:
!pip install -q PyMuPDF anthropic gspread
print("Dipendenze installate.")


## 2. Import e configurazione

Le costanti sotto replicano esattamente lo schema a 27 campi usato nell'estrazione
manuale (vedi CSV consegnato) — stesso identico prompt, stessa logica di retry.

In [ ]:
from __future__ import annotations

import base64
import json as _json
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import fitz  # PyMuPDF
import anthropic
import pandas as pd

CAMPI_RICHIESTI = [
    "nome_prodotto", "tipo_prodotto", "nome_indirizzo_fabbricante",
    "nome_indirizzo_importatore", "codice_a_barre", "lotto", "materiale",
    "impermeabilita", "dimensioni", "modalita_ricarica", "n_vibrazioni",
    "n_velocita", "n_modalita_suzione_tapping_rotazione", "strap_on_compatibile",
    "funzione_riscaldante", "telecomandato", "capacita_batteria", "garanzia",
    "simbolo_ce", "simbolo_ukca", "simbolo_raee", "simbolo_triman",
    "qr_code_junker", "simbolo_libretto_informativo", "codice_smaltimento_scatola",
    "codice_smaltimento_sacchetto", "contenuto",
]

MODEL = "claude-sonnet-4-6"
MATRIX_STANDARD = 2.0   # ~300 DPI, primo passaggio
MATRIX_HIRES = 8.0      # ~1200 DPI, retry mirato sul LOT
MAX_PAGES_PER_PDF = 3

print(f"{len(CAMPI_RICHIESTI)} campi configurati.")


## 3. Carica i PDF delle fustelle

Seleziona più file insieme (drag & drop funziona). Se preferisci lavorare da Google
Drive invece che caricarli ogni volta, decommenta il blocco "Drive" più sotto.

In [ ]:
from google.colab import files

uploaded = files.upload()
input_dir = Path("/content/fustelle")
input_dir.mkdir(exist_ok=True)
for nome, dati in uploaded.items():
    (input_dir / nome).write_bytes(dati)
print(f"{len(uploaded)} file caricati in {input_dir}")

# --- In alternativa, da Google Drive: -------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# input_dir = Path('/content/drive/MyDrive/fustelle')
# ---------------------------------------------------------------------------


## 4. Chiave API Claude
Non viene salvata: resta solo in memoria per questa sessione.

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Incolla la tua ANTHROPIC_API_KEY: ")
client = anthropic.Anthropic()
print("Client Claude pronto.")


## 5. Funzioni della pipeline
Stessa logica dello script `estrazione_fustelle.py` (dedup barcode, rilevamento PDF corrotti, retry ad alta risoluzione sul LOT).

In [ ]:
def barcode_da_nome_file(path: Path) -> Optional[str]:
    m = re.match(r"^(\d{10,14})_", path.name)
    return m.group(1) if m else None


def e_pdf_valido(path: Path) -> bool:
    """Rileva i PDF 'corrotti' (testo semplice rinominato .pdf) prima di
    farli fallire dentro fitz con uno stacktrace poco chiaro."""
    try:
        with open(path, "rb") as f:
            header = f.read(5)
        return header == b"%PDF-"
    except OSError:
        return False


def render_pagine(path: Path, matrix_scale: float = MATRIX_STANDARD) -> list[bytes]:
    immagini = []
    mat = fitz.Matrix(matrix_scale, matrix_scale)
    with fitz.open(path) as doc:
        for i, page in enumerate(doc):
            if i >= MAX_PAGES_PER_PDF:
                break
            pix = page.get_pixmap(matrix=mat)
            immagini.append(pix.tobytes("png"))
    return immagini


def render_hires_pagina0(path: Path) -> bytes:
    mat = fitz.Matrix(MATRIX_HIRES, MATRIX_HIRES)
    with fitz.open(path) as doc:
        pix = doc[0].get_pixmap(matrix=mat)
        return pix.tobytes("png")


PROMPT_ESTRAZIONE = """Sei un tecnico esperto di etichettatura conformita' prodotti
(regolamento REACH/RoHS/RAEE UE) che estrae dati da fustelle di packaging.

Ti mostro una o piu' immagini ad alta risoluzione della fustella (confezione stesa)
di UN singolo prodotto. Estrai ESATTAMENTE questi campi e rispondi SOLO con un
oggetto JSON valido, nessun testo prima o dopo, nessun blocco markdown:

{campi}

REGOLE:
- Se un campo non e' applicabile al prodotto (es. "n_vibrazioni" per un dildo senza
  batteria), usa la stringa "N/D (nessuna batteria)" o "N/D" a seconda del contesto.
- Se un campo e' applicabile ma il testo e' troppo piccolo/sfocato per essere letto
  con certezza, usa la stringa "DA VERIFICARE (zoom)" - non inventare mai un valore.
- "nome_indirizzo_fabbricante" e "nome_indirizzo_importatore" vanno presi dal testo
  "Prodotto e importato da..." stampato sulla fustella; se coincidono, ripeti lo
  stesso valore in entrambi i campi.
- I simboli booleani (simbolo_ce, simbolo_ukca, simbolo_raee, simbolo_triman,
  qr_code_junker, simbolo_libretto_informativo, strap_on_compatibile,
  funzione_riscaldante, telecomandato) vanno valorizzati con "Si" o "No", mai N/D,
  a meno che l'area dove dovrebbero comparire non sia visibile/leggibile.
- codice_smaltimento_scatola e codice_smaltimento_sacchetto: riporta il codice
  materiale (es. PAP, CPE, PET) come stampato nel triangolo di riciclo.
- barcode: leggilo direttamente dal codice a barre stampato, cifra per cifra.
"""


def costruisci_prompt() -> str:
    campi_fmt = ",\n".join(f'  "{c}": "..."' for c in CAMPI_RICHIESTI)
    return PROMPT_ESTRAZIONE.format(campi="{\n" + campi_fmt + "\n}")


def chiama_vision(immagini_png: list[bytes]) -> dict:
    content = []
    for img_bytes in immagini_png:
        content.append({
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": "image/png",
                "data": base64.b64encode(img_bytes).decode("utf-8"),
            },
        })
    content.append({"type": "text", "text": costruisci_prompt()})

    resp = client.messages.create(
        model=MODEL, max_tokens=2000,
        messages=[{"role": "user", "content": content}],
    )
    testo = "".join(b.text for b in resp.content if b.type == "text")
    testo = testo.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return _json.loads(testo)


@dataclass
class RisultatoEstrazione:
    dati: dict = field(default_factory=dict)
    stato: str = "Errore"
    motivo: str = ""
    file_pdf: str = ""


def estrai_uno(path: Path) -> RisultatoEstrazione:
    ris = RisultatoEstrazione(file_pdf=path.name)

    if not e_pdf_valido(path):
        testo_grezzo = path.read_text(errors="ignore") if path.exists() else ""
        ris.dati = {c: "N/D (PDF corrotto)" for c in CAMPI_RICHIESTI}
        ris.dati["codice_a_barre"] = barcode_da_nome_file(path) or "N/D"
        ris.dati["nome_prodotto"] = path.stem.split("_", 2)[-1].replace("_", " ")
        m = re.search(r"LOT:\s*([A-Za-z0-9]+)", testo_grezzo)
        if m:
            ris.dati["lotto"] = m.group(1)
        ris.stato = "Parziale"
        ris.motivo = "PDF corrotto (file di testo semplice, non vettoriale)"
        return ris

    try:
        immagini = render_pagine(path)
        dati = chiama_vision(immagini)

        if str(dati.get("lotto", "")).startswith("DA VERIFICARE"):
            img_hires = render_hires_pagina0(path)
            dati_hires = chiama_vision([img_hires])
            if not str(dati_hires.get("lotto", "")).startswith("DA VERIFICARE"):
                dati["lotto"] = dati_hires["lotto"]

        mancanti = [c for c in CAMPI_RICHIESTI if c not in dati]
        ris.dati = dati
        if mancanti:
            ris.stato = "Parziale"
            ris.motivo = f"Campi mancanti: {mancanti}"
            return ris

        valori_dubbi = [v for v in dati.values() if str(v).startswith("DA VERIFICARE")]
        ris.stato = "Parziale" if valori_dubbi else "Completo"
        if valori_dubbi:
            ris.motivo = "Uno o piu' campi illeggibili anche dopo il retry ad alta risoluzione"
        return ris

    except Exception as exc:
        ris.motivo = f"Eccezione: {exc}"
        return ris


print("Funzioni pipeline pronte.")


## 6. Esecuzione sul batch caricato

Dedup automatico per barcode (letto dal nome file `<barcode>_<dimensioni>_<nome>.pdf`):
se lanci di nuovo la cella su una cartella con file già visti, non li riprocessa.

In [ ]:
tutti_i_pdf = sorted(input_dir.glob("*.pdf"))
print(f"{len(tutti_i_pdf)} file trovati in {input_dir}")

risultati = []
barcode_visti = set()

for i, path in enumerate(tutti_i_pdf, 1):
    bc = barcode_da_nome_file(path)
    if bc and bc in barcode_visti:
        print(f"[{i}/{len(tutti_i_pdf)}] barcode {bc} duplicato, salto {path.name}")
        continue
    print(f"[{i}/{len(tutti_i_pdf)}] {path.name}")
    ris = estrai_uno(path)
    if bc:
        barcode_visti.add(bc)
    risultati.append(ris)
    time.sleep(0.5)

completi = sum(1 for r in risultati if r.stato == "Completo")
parziali = sum(1 for r in risultati if r.stato == "Parziale")
errori = sum(1 for r in risultati if r.stato == "Errore")
print(f"\nFATTO. Completo={completi} Parziale={parziali} Errore={errori}")


## 7. Tabella risultati + download CSV

In [ ]:
righe = []
for r in risultati:
    riga = {c: r.dati.get(c, "N/D") for c in CAMPI_RICHIESTI}
    riga["stato_estrazione"] = r.stato + (f" - {r.motivo}" if r.motivo else "")
    riga["file_origine"] = r.file_pdf
    righe.append(riga)

df = pd.DataFrame(righe)
df.to_csv("/content/estrazione_pack.csv", index=False)
print("Salvato in /content/estrazione_pack.csv")
df


In [ ]:
from google.colab import files as _files
_files.download("/content/estrazione_pack.csv")


## 8. (Opzionale) Scrivi direttamente su Google Sheet

Ti verrà chiesto di autenticarti col tuo account Google — nessun file di
credenziali da preparare, funziona perché sei già dentro Colab.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

SHEET_ID = "INCOLLA_QUI_L_ID_DEL_TUO_GOOGLE_SHEET"  # dall'URL: .../d/<ID>/edit
SHEET_TAB = "estrazione_pack"

sh = gc.open_by_key(SHEET_ID)
try:
    ws = sh.worksheet(SHEET_TAB)
except gspread.WorksheetNotFound:
    ws = sh.add_worksheet(title=SHEET_TAB, rows=200, cols=len(CAMPI_RICHIESTI) + 2)
    ws.append_row(CAMPI_RICHIESTI + ["stato_estrazione", "file_origine"])

barcode_col = CAMPI_RICHIESTI.index("codice_a_barre") + 1
gia_presenti = set(ws.col_values(barcode_col)[1:])

scritte = 0
for riga in righe:
    if riga["codice_a_barre"] in gia_presenti:
        continue
    ws.append_row([riga.get(c, "N/D") for c in CAMPI_RICHIESTI] + [riga["stato_estrazione"], riga["file_origine"]],
                  value_input_option="USER_ENTERED")
    scritte += 1

print(f"{scritte} nuove righe scritte su Google Sheet (saltati eventuali duplicati per barcode).")
